# 04. Data ML Agent — 파일·샘플 기반 추론
**Day 14**

> `data/` 폴더의 CSV를 Agent가 **파일명 + 샘플 번호**로 읽어 ML 추론을 수행합니다.
>
> 예: *"failure_equipment.csv 3번 샘플이 이상한지 XGBoost로 확인해봐"*

---
#### **<실습 태스크>**

| 구간 | 직접 할 일 |
|------|-----------|
| ✏️ 실습 1 | `preview_data_sample`으로 다른 샘플 번호 조회 |
| ✏️ 실습 2 | `logistic_predict_from_data` 직접 호출 |
| ✏️ 실습 3 | Agent `system` 프롬프트에 규칙 1줄 추가 |
| ✏️ 실습 4 | 자연어 질문 1개 직접 작성 |
| ✏️ 실습 5 | 멀티턴 후속 질문 직접 작성 |
| ✏️ 실습 6 (선택) | 14.1 Tool과 `DATA_TOOLS` 합치기 |

막히면 위 데모 셀·`ml_tools/data_tools.py`를 참고하되, **먼저 스스로** 채워 봅니다.

---
## 0. 설치 & 경로

In [ ]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-classic scikit-learn xgboost joblib

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
DATA_DIR = WORKDIR / 'data'
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print('WORKDIR:', WORKDIR)
print('DATA_DIR:', DATA_DIR)
for p in sorted(DATA_DIR.glob('*.csv')):
    print(' -', p.name)

---
## 1. data/ 폴더 구조

| 파일 | 유형 | 설명 |
|------|------|------|
| `failure_equipment.csv` | 설비 고장 | Temperature, Humidity, Operator, Measure1~15 … **Failure**(Yes/No) |
| `anomaly_sensor.csv` | 이상 탐지 | 2D 센서 (`0`, `1`) + 라벨 **Y** (1=정상, -1=이상) |

- **샘플 번호**는 **1부터** (1번 = CSV 첫 데이터 행)
- Agent Tool은 `ml_tools/data_tools.py`에 정의

---
## Step 1. Data Tool 불러오기

In [ ]:
from ml_tools import DATA_TOOLS, list_data_files, preview_data_sample, xgboost_predict_from_data

print('Data Tool 목록:')
for t in DATA_TOOLS:
    print(f' - {t.name}')

In [ ]:
print(list_data_files.invoke({}))

---
## ✏️ 실습 1. 다른 샘플 미리보기

`anomaly_sensor.csv`에서 **본인이 정한 샘플 번호**(1~100)로 `preview_data_sample`을 호출하세요.

**관찰:** 컬럼 `0`, `1`, `Y` 값과 실제 라벨 의미(1=정상, -1=이상)

In [ ]:
# ── 여기에 작성 ──
# my_sample = ...
# print(preview_data_sample.invoke({
#     'filename': 'anomaly_sensor.csv',
#     'sample_index': my_sample,
# }))

---
## Step 2. Tool 단독 테스트

In [ ]:
print(preview_data_sample.invoke({
    'filename': 'failure_equipment.csv',
    'sample_index': 3,
}))

In [ ]:
print(xgboost_predict_from_data.invoke({
    'filename': 'failure_equipment.csv',
    'sample_index': 3,
}))

---
## ✏️ 실습 2. 로지스틱 예측 직접 호출

`logistic_predict_from_data`로 `failure_equipment.csv` **10번** 샘플을 예측하고, 위 XGBoost 3번 결과와 **확률 차이**를 비교해 보세요.

**힌트:** `from ml_tools import logistic_predict_from_data`

In [ ]:
# ── 여기에 작성 ──
# from ml_tools import logistic_predict_from_data
#
# print(logistic_predict_from_data.invoke({
#     'filename': 'failure_equipment.csv',
#     'sample_index': 10,
# }))

---
## Step 3. AgentExecutor — Data Tool 통합

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = DATA_TOOLS
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 data/ 폴더 CSV를 분석하는 제조 AI입니다.

사용 가능한 data/ 파일:
- failure_equipment.csv — 설비 고장 예측 (Failure 컬럼)
- anomaly_sensor.csv — 2D 센서 이상 탐지 (Y 컬럼)

도구 선택:
- 파일 목록 → list_data_files
- 샘플 원본 확인 → preview_data_sample(filename, sample_index)
- 고장 XGBoost → xgboost_predict_from_data(filename, sample_index)
- 고장 로지스틱 → logistic_predict_from_data(filename, sample_index)
- 이상 탐지 → anomaly_detect_from_data(filename, sample_index)

사용자가 "3번 샘플"이라고 하면 sample_index=3 입니다.
파일명이 없으면 list_data_files로 확인하거나 질문 맥락에서 추론하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## ✏️ 실습 3. 시스템 프롬프트 수정

Step 3 `prompt` **system**에 아래 규칙을 **한 줄** 추가한 뒤 Agent를 다시 만드세요.

> *"예측 전에 항상 preview_data_sample로 해당 샘플 원본을 먼저 확인하세요."*

In [ ]:
# ── 여기에 작성 ──
# (Step 3 prompt 복사 후 system 문자열 수정)
# agent = create_tool_calling_agent(llm, agent_tools, prompt)
# executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## Step 4. 자연어 데모

In [ ]:
r = executor.invoke({
    'input': 'failure_equipment.csv 3번 샘플이 이상한지 XGBoost로 확인해봐',
    'chat_history': [],
})
print(r['output'])

In [ ]:
r = executor.invoke({
    'input': 'anomaly_sensor.csv 5번 샘플 정상인지 이상 탐지해줘',
    'chat_history': [],
})
print(r['output'])

In [ ]:
r = executor.invoke({
    'input': 'failure_equipment.csv 25번은 로지스틱이랑 XGBoost 둘 다 예측해봐',
    'chat_history': [],
})
print(r['output'])

---
## ✏️ 실습 4. 자연어 질문 직접 작성

Step 4 데모와 **다른** 파일·샘플·모델 조합으로 `executor.invoke`를 호출하세요.

**예시:** *"anomaly_sensor.csv 20번 이상 탐지해줘"*, *"failure_equipment.csv 50번 로지스틱으로"*

In [ ]:
# ── 여기에 작성 ──
# my_question = '...'
# r = executor.invoke({'input': my_question, 'chat_history': []})
# print(r['output'])

---
## Step 5. 멀티턴

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('data 폴더에 뭐 있어?'))
print('2:', session.ask('failure_equipment.csv 10번 XGBoost로 봐줘'))

---
## ✏️ 실습 5. 멀티턴 후속 질문

`ChatSession`으로 1턴 *"data 폴더 목록"* 후, 2턴에서 **특정 파일·샘플 예측**을 이어가는 질문을 직접 작성하세요.

In [ ]:
# ── 여기에 작성 ──
# session2 = ChatSession(executor)
# print('1:', session2.ask('...'))
# print('2:', session2.ask('...'))  # 1턴 맥락을 이어가는 질문

---
## Step 6. (선택) 14.1 + 14.3과 통합

RAG · 웹 · VLM · Data ML을 한 Agent에:

### ✏️ 실습 6

1. **14.1 노트북 실행** 후 `search_pdf_tool`, `web_search`를 가져옵니다.
2. 아래 셀에서 `agent_tools`에 `DATA_TOOLS`를 합치고, `system` 프롬프트에 PDF/웹/데이터 Tool 용도를 명시하세요.

In [ ]:
# ── 여기에 작성 (14.1 실행 후) ──
# from ml_tools import DATA_TOOLS
#
# agent_tools = [
#     search_pdf_tool,
#     web_search,
#     *DATA_TOOLS,
# ]
#
# combined_prompt = ChatPromptTemplate.from_messages([
#     ('system', '''... PDF / 웹 / data Tool 용도 ...'''),
#     MessagesPlaceholder('chat_history', optional=True),
#     ('human', '{input}'),
#     MessagesPlaceholder('agent_scratchpad'),
# ])
#
# combined_agent = create_tool_calling_agent(llm, agent_tools, combined_prompt)
# combined_executor = AgentExecutor(agent=combined_agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## 정리

```
14일차_실습/
├── data/                          ← Agent가 읽는 CSV
│   ├── failure_equipment.csv
│   └── anomaly_sensor.csv
└── ml_tools/
    ├── data_tools.py              ← list / preview / predict from file
    └── models/*.joblib
```

새 CSV를 `data/`에 넣으면 `list_data_files`로 바로 확인할 수 있습니다.